In [294]:
import pandas as pd
from konlpy.tag import Okt
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

In [295]:
df = pd.read_json('../data_git/data_NLP/1-1.여성의류(196).json')
df

,Index,RawText,Source,Domain,MainCategory,ProductName,Syllable,Word,GeneralPolarity,Aspects
0,1024338,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...,SNS,패션,여성의류,OO 경량 다운 자켓,513,121,1,"[{'Aspect': '디자인', 'SentimentText': '딱 기본 스타일인..."
1,1024477,드디어 겨울이 찾아왔네요. 이제부터 슬슬 겨울 패딩 장만하셔야지요? 패딩 소개해 드...,SNS,패션,여성의류,OO 아** 구스코트,464,105,1,"[{'Aspect': '사이즈', 'SentimentText': '저는 블랙90 사..."
2,1025044,오늘도 정말 춥네요... 롱패딩 찾고 계신 분을 위한 후기 공유합니다. 키 158...,SNS,패션,여성의류,OO 아** 구스코트,314,78,1,"[{'Aspect': '사이즈', 'SentimentText': '키 158센티로에..."
3,1025046,이웃님들 오늘도 안녕하신가요? 오늘은 따끈한 신상 패딩 후기 올려봅니다~~ 겨울이...,SNS,패션,여성의류,OO 아** 구스코트,307,74,1,"[{'Aspect': '색상', 'SentimentText': '흰색 패딩이 너무나..."
4,1025071,OOO 구스로 소문난 OO의 롱패딩~ 한번 구경 가봐요. 일단 보는 순간 고급스럽...,SNS,패션,여성의류,OO 아** 구스코트,279,68,1,"[{'Aspect': '소재', 'SentimentText': ' 일단 보는 순간 ..."
...,...,...,...,...,...,...,...,...,...,...
118,1028153,요즘 출 퇴근할 때 너무 추워서 목폴라 상품을 보고 있다가 마음에 드는 옷을 발견했...,SNS,패션,여성의류,OO 여성용 목폴라 티셔츠,294,75,1,"[{'Aspect': '두께', 'SentimentText': '저는 두께가 얇은 ..."
119,1028154,오늘 소개 해 드릴 옷은 겨울에 따뜻하게 입을 수 있는 원피스 하나 소개 해 드리려...,SNS,패션,여성의류,OO 케** 반집업니트원피스,296,75,1,"[{'Aspect': '기능', 'SentimentText': '겨울에 따뜻하게 입..."
120,1028155,기존 목폴라 티셔츠가 다 늘어져 구입하려고 찾아보던 중 알게 된 OOO 비네츠 폴라...,SNS,패션,여성의류,OO 비** 폴라티,298,71,1,"[{'Aspect': '소재', 'SentimentText': '이 목폴라 티셔츠는..."
121,1028156,몸에 딱 맞는 바지 찾기가 너무 힘든데 제가 원하던 바지를 찾아서 글을 남깁니다. ...,SNS,패션,여성의류,OO 프리미엄 팬츠,318,78,1,"[{'Aspect': '소재', 'SentimentText': '일단 원단이 너무 ..."


## 문제
- 데이터프레임에서 'Aspects' 컬럼의 데이터들을 이용하여 분류 모델을 생성하려 한다.
- SentimentText 텍스트를 이용하여 'Aspect', 'SentimentPolarity'의 값들을 예측하는 모델을 생성
1. df 에서 'Aspects' 데이터를 추출
2. SentimentText 데이터는 문자형으로 이루어져있으니 학습에 대한 데이터의 형태로 변환 (문자의 데이터를 숫자형 데이터로) -> 토큰화(okt), 벡터화(TF-IDF)
3. 종속 변수는 'Aspect', 'SentimentPolarity'
4. 분류모델(LinearSVC) random_state만 42로 고정
5. 테스트를 이용하여 분류가 잘되고 있는가? 정확도만 확인 (197 데이터를 로드하여 정확도 계산)
- 벡터화, 모델링 파이프라인으로 연결해서 사용

In [296]:
test_text = [
    '색상이 마음에 든다',
    '설명에 비해 옷이 두껍진 않다',
    '길이가 너무 길지도 않고 짧지도 않다'
]

In [297]:
data = df['Aspects'].values

In [298]:
data_list = []
for dic in data:
    for x in dic:
        data_list.append(x)

In [299]:
df_aspect = pd.DataFrame(data_list)

In [300]:
# strip
df_aspect.iloc[:, :2] = df_aspect.iloc[:, :2].map(
    lambda x : x.strip()
)

In [301]:
df_aspect

,Aspect,SentimentText,SentimentWord,SentimentPolarity
0,디자인,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서,11,1
1,두께,이것만 입기엔 얇지만,3,-1
2,기능,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.,12,1
3,색상,색상도 디자인도 무난해서,3,0
4,디자인,디자인도 무난해서,2,0
...,...,...,...,...
912,길이,길이감도 너무 짧거나 애매한 길이가 아니고 적당합니다.,7,1
913,활용성,트레이닝 세트나 데님 스커트 후드 원피스 등에도 잘 어울립니다.,9,1
914,디자인,전체적으로 고급스러움이 잘 녹아있는 디자인으로 되어 있고,7,1
915,품질,퀄리티도 넘 휼륭합니다.,3,1


In [292]:
le = LabelEncoder()
df_aspect['Aspect'] = le.fit_transform(df_aspect['Aspect'])

In [293]:
df_aspect

,Aspect,SentimentText,SentimentWord,SentimentPolarity
0,4,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서,11,1
1,3,이것만 입기엔 얇지만,3,-1
2,1,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.,12,1
3,8,색상도 디자인도 무난해서,3,0
4,4,디자인도 무난해서,2,0
...,...,...,...,...
912,2,길이감도 너무 짧거나 애매한 길이가 아니고 적당합니다.,7,1
913,16,트레이닝 세트나 데님 스커트 후드 원피스 등에도 잘 어울립니다.,9,1
914,4,전체적으로 고급스러움이 잘 녹아있는 디자인으로 되어 있고,7,1
915,14,퀄리티도 넘 휼륭합니다.,3,1


In [172]:
X = df_aspect['SentimentText'].values
Y1 = df_aspect['Aspect'].values
Y2 = df_aspect['SentimentPolarity'].values

In [173]:
okt = Okt()
def tokenize(text):
    return okt.morphs(text)

# 토큰화, 벡터화
vectorizer1 = TfidfVectorizer(
    tokenizer=tokenize
)
vectorizer2 = TfidfVectorizer(
    tokenizer=tokenize
)
# 모델
svc1 = LinearSVC(random_state=42)
svc2 = LinearSVC(random_state=42)

In [174]:
# aspect 추측모델
pipe1 = Pipeline(
    [
        ('vector' , vectorizer1),
        ('model', svc1)
    ]
)

In [175]:
# SentimentPolarity 추측 모델
pipe2 = Pipeline(
    [
        ('vector', vectorizer2),
        ('model', svc2)
    ]
)

In [176]:
pipe1.fit(X, Y1)

/Users/eunseo/Documents/data_boot/venv/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,steps,"[('vector', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,<function tok...t 0x177a61440>


In [177]:
pred1 = pipe1.predict(test_text)

In [178]:
le.inverse_transform(pred1)

array(['색상', '디자인', '길이'], dtype=object)

In [179]:
pipe2.fit(X, Y2)

/Users/eunseo/Documents/data_boot/venv/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,steps,"[('vector', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,<function tok...t 0x177a61440>


In [180]:
pred2 = pipe2.predict(test_text)

In [181]:
pred2

array(['1', '1', '1'], dtype=object)

### 방법2

In [182]:
df_data = pd.DataFrame(data_list)

In [183]:
df_data['SentimentPolarity'].value_counts()

SentimentPolarity
1     808
-1     81
0      28
Name: count, dtype: int64

In [184]:
df_data['SentimentPolarity'] == 1

0      False
1      False
2      False
3      False
4      False
       ...  
912    False
913    False
914    False
915    False
916    False
Name: SentimentPolarity, Length: 917, dtype: bool

In [185]:
df_data['Aspect_Polarity'] = df_data['Aspect'] + df_data['SentimentPolarity'].astype(str)
df_data['Aspect_Polarity'].value_counts()

Aspect_Polarity
디자인1      102
기능1       100
소재1        86
활용성1       84
색상1        66
핏1         61
가격1        52
착용감1       46
사이즈1       43
길이1        29
무게1        24
촉감1        23
제품구성1      22
신축성1       21
품질1        21
두께1        20
사이즈-1      15
디자인-1      13
마감1         8
색상-1        8
무게-1        7
가격-1        6
두께-1        6
사이즈0        6
디자인0        6
소재-1        5
색상0         5
품질-1        4
길이-1        4
핏-1         3
마감-1        3
두께0         2
제품구성0       2
소재0         2
길이0         2
활용성-1       2
촉감-1        2
품질0         1
제품구성-1      1
활용성0        1
촉감0         1
착용감-1       1
기능-1        1
Name: count, dtype: int64

In [186]:
df_data

,Aspect,SentimentText,SentimentWord,SentimentPolarity,Aspect_Polarity
0,디자인,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서,11,1,디자인1
1,두께,이것만 입기엔 얇지만,3,-1,두께-1
2,기능,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.,12,1,기능1
3,색상,색상도 디자인도 무난해서,3,0,색상0
4,디자인,디자인도 무난해서,2,0,디자인0
...,...,...,...,...,...
912,길이,길이감도 너무 짧거나 애매한 길이가 아니고 적당합니다.,7,1,길이1
913,활용성,트레이닝 세트나 데님 스커트 후드 원피스 등에도 잘 어울립니다.,9,1,활용성1
914,디자인,전체적으로 고급스러움이 잘 녹아있는 디자인으로 되어 있고,7,1,디자인1
915,품질,퀄리티도 넘 휼륭합니다.,3,1,품질1


In [187]:
X = df_data['SentimentText']
Y = df_data['Aspect_Polarity']

le2 = LabelEncoder()
Y_le = le2.fit_transform(Y)

In [188]:
okt = Okt()
def tokenize(text):
    return okt.morphs(text)

# 토큰화, 벡터화
vectorizer = TfidfVectorizer(
    tokenizer=tokenize
)

# 모델
svc = LinearSVC(random_state=42)

In [189]:
pipe = Pipeline(
    [
        ('vector', vectorizer),
        ('model', svc)
    ]
)

In [190]:
pipe.fit(X, Y_le)

/Users/eunseo/Documents/data_boot/venv/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,steps,"[('vector', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,<function tok...t 0x1772b85e0>


In [191]:
pred = pipe.predict(test_text)
pred

array([22, 12,  6])

In [192]:
le2.inverse_transform(pred)

array(['색상1', '디자인1', '길이1'], dtype=object)

- 정확도 평가

In [244]:
df_197 = pd.read_json('../data_git/data_NLP/1-1.여성의류(196).json')

In [245]:
data = df_197['Aspects'].values

In [246]:
data_list_197 = []
for dic in data:
    for x in dic:
        data_list_197.append(x)

df_aspect_197 = pd.DataFrame(data_list_197)

In [247]:
df_aspect_197

,Aspect,SentimentText,SentimentWord,SentimentPolarity
0,디자인,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서,11,1
1,두께,이것만 입기엔 얇지만,3,-1
2,기능,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.,12,1
3,색상,색상도 디자인도 무난해서,3,0
4,디자인,디자인도 무난해서,2,0
...,...,...,...,...
912,길이,길이감도 너무 짧거나 애매한 길이가 아니고 적당합니다.,7,1
913,활용성,트레이닝 세트나 데님 스커트 후드 원피스 등에도 잘 어울립니다.,9,1
914,디자인,전체적으로 고급스러움이 잘 녹아있는 디자인으로 되어 있고,7,1
915,품질,퀄리티도 넘 휼륭합니다.,3,1


In [248]:
df_aspect_197['Aspect_SentimentPolarity'] = df_aspect_197["Aspect"] + df_aspect_197["SentimentPolarity"].astype(str)
df_aspect_197

,Aspect,SentimentText,SentimentWord,SentimentPolarity,Aspect_SentimentPolarity
0,디자인,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서,11,1,디자인1
1,두께,이것만 입기엔 얇지만,3,-1,두께-1
2,기능,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.,12,1,기능1
3,색상,색상도 디자인도 무난해서,3,0,색상0
4,디자인,디자인도 무난해서,2,0,디자인0
...,...,...,...,...,...
912,길이,길이감도 너무 짧거나 애매한 길이가 아니고 적당합니다.,7,1,길이1
913,활용성,트레이닝 세트나 데님 스커트 후드 원피스 등에도 잘 어울립니다.,9,1,활용성1
914,디자인,전체적으로 고급스러움이 잘 녹아있는 디자인으로 되어 있고,7,1,디자인1
915,품질,퀄리티도 넘 휼륭합니다.,3,1,품질1


In [249]:
le = LabelEncoder()
df_aspect_197['Aspect'] = le.fit_transform(df_aspect_197['Aspect'])

In [250]:
X_test = df_aspect_197['SentimentText']
Y_test_aspect = df_aspect_197['Aspect']
Y_test_sen = df_aspect_197['SentimentPolarity']
Y_test = df_aspect_197['Aspect_SentimentPolarity']

In [251]:
# 방법 1
pred1_197 = pipe1.predict(X_test)
pred2_197 = pipe2.predict(X_test)

In [252]:
acc1 = accuracy_score(pred1_197, Y_test_aspect)
acc2 = accuracy_score(pred2_197, Y_test_sen)

print(f"aspect_accuracy: {round(acc1, 4)} \nsentiment polarity_accuracy : {round(acc2, 4)}")

aspect_accuracy: 0.9989 
sentiment polarity_accuracy : 1.0


In [253]:
# 방법 2
pred_197 = pipe.predict(X_test)

In [255]:
pred_197_le = le2.inverse_transform(pred_197)

In [261]:
acc = accuracy_score(pred_197_le, Y_test)
print(f"accuracy_onepipe : {round(acc, 4)}")

accuracy_onepipe : 0.9978


- 방법3

In [302]:
pd.DataFrame(df['Aspects'].sum())

,Aspect,SentimentText,SentimentWord,SentimentPolarity
0,디자인,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서,11,1
1,두께,이것만 입기엔 얇지만,3,-1
2,기능,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.,12,1
3,색상,색상도 디자인도 무난해서,3,0
4,디자인,디자인도 무난해서,2,0
...,...,...,...,...
912,길이,길이감도 너무 짧거나 애매한 길이가 아니고 적당합니다.,7,1
913,활용성,트레이닝 세트나 데님 스커트 후드 원피스 등에도 잘 어울립니다.,9,1
914,디자인,전체적으로 고급스러움이 잘 녹아있는 디자인으로 되어 있고,7,1
915,품질,퀄리티도 넘 휼륭합니다.,3,1


In [303]:
# 종속변수가 2개인 경우 일반적으로 사용하는 객체
from sklearn.multioutput import MultiOutputClassifier

In [304]:
# 종속 변수의 크기가 (2000, 2)
# 첫번쨰 종속의 데이터를 이용하여 fit() -> predict
# 두번째 종속의 데이터를 이용하여 fit() -> predict
# 위의 2개의 작업을 병렬로 처리

In [320]:
# 분류 모델을 생성
svc = LinearSVC(random_state=42)
# 멀티 아웃 모델을 생성
multi_model = MultiOutputClassifier(svc)
# 파이프라인 생성
pipe_multi = Pipeline(
    [
        ('vector', vectorizer),
        ('model', multi_model)
    ]
)

In [323]:
# 멀티 모델 종속은 2차원 그대로 사용
X = df_aspect['SentimentText'].values
Y = df_aspect[['Aspect', 'SentimentPolarity']].values

In [ ]:
pipe_multi.fit(X,Y)

In [ ]:
pred_multi = pipe_multi.predict(X_test)

In [ ]:
pred_multi

array([['디자인', '1'],
       ['두께', '-1'],
       ['기능', '1'],
       ...,
       ['디자인', '1'],
       ['품질', '1'],
       ['디자인', '1']], shape=(917, 2), dtype=object)

In [325]:
# 문단이 장문인 데이터에서 문장별로 나눠주기
from konlpy.tag import Kkma

In [328]:
text = df.loc[0, 'RawText']

In [329]:
kkma = Kkma()

In [331]:
texts = kkma.sentences(text)

In [ ]:
pred = pipe_multi.predict(texts)

In [ ]:
le.inverse_transform(pred[:, 0])